# 🌸 TohoStudio: Google Colab GPU メディア生成サーバー（動画・画像・音声・SE）

このノートブックは、Google Colab の無料 NVIDIA GPU（T4 / VRAM 16GB）を活用し、**「高画質イラスト」「アニメ動画（MP4）」「BGM」「効果音（SE）」**の4大メディア生成を手元の Mac に提供するサーバーです。

### 安定化チューニング
- 3つの巨大モデルを同時にロードしてもメモリ不足（VRAM OOM）を起こさないよう、**CPU オフロード最適化** を適用しています。

In [ ]:
# [1/4] GPU の確認
!nvidia-smi

In [ ]:
# [2/4] 依存ライブラリと Cloudflare Tunnel のインストール
!pip install -q fastapi uvicorn pydantic diffusers transformers accelerate torch scipy soundfile imageio[ffmpeg]

# Cloudflare Tunnel (cloudflared) の取得
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ ライブラリとトンネルツールの準備が完了しました！")

In [ ]:
# [3/4] FastAPI サーバープログラムの作成 (VRAMメモリ節約・完全安定版)
server_code = '''
import io, os, tempfile, torch, scipy.io.wavfile, gc
from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel
from typing import Optional
from diffusers import AutoPipelineForText2Image, AnimateDiffPipeline, MotionAdapter, DDIMScheduler
from diffusers.utils import export_to_video
from transformers import AutoProcessor, MusicgenForConditionalGeneration

app = FastAPI(title="TohoStudio Colab GPU Media Bridge", version="2.1.0")
device = "cuda" if torch.cuda.is_available() else "cpu"

# グローバル変数（オンデマンドで初期化してVRAMを節約）
pipe_img = None
music_processor = None
music_model = None
pipe_video = None

def get_image_pipe():
    global pipe_img
    if pipe_img is None:
        print("Loading SD-Turbo Image Pipeline...")
        pipe_img = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sd-turbo",
            torch_dtype=torch.float16,
            variant="fp16"
        ).to(device)
    return pipe_img

def get_music_pipe():
    global music_processor, music_model
    if music_model is None:
        print("Loading MusicGen Pipeline...")
        music_processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
        music_model = MusicgenForConditionalGeneration.from_pretrained(
            "facebook/musicgen-small",
            torch_dtype=torch.float16
        ).to(device)
    return music_processor, music_model

def get_video_pipe():
    global pipe_video
    if pipe_video is None:
        print("Loading AnimateDiff Video Pipeline...")
        adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
        pipe_video = AnimateDiffPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            motion_adapter=adapter,
            torch_dtype=torch.float16
        )
        pipe_video.scheduler = DDIMScheduler.from_config(pipe_video.scheduler.config, beta_schedule="linear", steps_offset=1)
        pipe_video.enable_model_cpu_offload()
    return pipe_video

@app.get("/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    vram_free = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    return {
        "status": "READY",
        "gpu": gpu_name,
        "vram_free_gb": round(vram_free, 2),
        "capabilities": ["image", "video", "bgm", "se"]
    }

class MediaPromptRequest(BaseModel):
    prompt: str
    duration_seconds: Optional[float] = 8.0

# [1] 画像生成
@app.post("/v1/generate/image")
def generate_image(req: MediaPromptRequest):
    pipe = get_image_pipe()
    image = pipe(prompt=req.prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return Response(content=buf.getvalue(), media_type="image/png")

# [2] BGM 音楽生成
@app.post("/v1/generate/music")
@app.post("/v1/generate/bgm")
def generate_bgm(req: MediaPromptRequest):
    proc, model = get_music_pipe()
    inputs = proc(text=[req.prompt], padding=True, return_tensors="pt").to(device)
    max_tokens = min(750, int(req.duration_seconds * 50))
    audio_values = model.generate(**inputs, max_new_tokens=max_tokens)
    sampling_rate = model.config.audio_encoder.sampling_rate
    wav_data = audio_values[0, 0].cpu().numpy()
    buf = io.BytesIO()
    scipy.io.wavfile.write(buf, rate=sampling_rate, data=wav_data)
    return Response(content=buf.getvalue(), media_type="audio/wav")

# [3] 効果音 (SE) 生成
@app.post("/v1/generate/se")
def generate_se(req: MediaPromptRequest):
    proc, model = get_music_pipe()
    se_prompt = f"sound effect, foley, cinematic SFX: {req.prompt}"
    inputs = proc(text=[se_prompt], padding=True, return_tensors="pt").to(device)
    duration = min(4.0, max(1.0, req.duration_seconds if req.duration_seconds <= 4.0 else 2.5))
    max_tokens = int(duration * 50)
    audio_values = model.generate(**inputs, max_new_tokens=max_tokens)
    sampling_rate = model.config.audio_encoder.sampling_rate
    wav_data = audio_values[0, 0].cpu().numpy()
    buf = io.BytesIO()
    scipy.io.wavfile.write(buf, rate=sampling_rate, data=wav_data)
    return Response(content=buf.getvalue(), media_type="audio/wav")

# [4] アニメ動画 (MP4) 生成
@app.post("/v1/generate/video")
def generate_video(req: MediaPromptRequest):
    pipe = get_video_pipe()
    output = pipe(
        prompt=req.prompt,
        negative_prompt="worst quality, low quality, deformed, blurry",
        num_frames=16,
        guidance_scale=7.5,
        num_inference_steps=20,
    )
    frames = output.frames[0]
    temp_file = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)
    temp_path = temp_file.name
    temp_file.close()

    export_to_video(frames, temp_path, fps=8)
    with open(temp_path, "rb") as f:
        video_data = f.read()
    os.remove(temp_path)
    return Response(content=video_data, media_type="video/mp4")
'''

with open("app.py", "w") as f:
    f.write(server_code)
print("✅ 安定化サーバーコード（app.py）を書き出しました。")

In [ ]:
# [4/4] サーバー起動と外部トンネルURLの発行
import subprocess, time, re

# 既存のプロセスを終了
!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

# uvicorn サーバーを起動（オンデマンドロードのため数秒で起動します）
print("サーバーを起動しています...")
server_proc = subprocess.Popen(["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for _ in range(30):
    line = server_proc.stdout.readline()
    if line:
        print(line.strip())
    if "Application startup complete" in line:
        break
    time.sleep(0.5)

# Cloudflare Tunnel を起動して一時 URL を取得
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if public_url:
    print("\n===================================================================")
    print("🎉 Colab GPU 4大メディア生成サーバーが正常に起動しました！")
    print(f"👉 接続先 URL: {public_url}")
    print("===================================================================\n")
else:
    print("トンネルURLの取得に失敗しました。")